In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import cv2

In [25]:
def preprocess_image(img, img_size=(64, 40)):
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32) / 255.0

    return img

In [42]:
model_path = os.path.join('sprite_mask_unet.keras')
model = load_model(model_path)

def extract_foreground(img):
    img = preprocess_image(img)
    pred_mask = model.predict(img[np.newaxis, ...])[0]
    pred_mask_bin = (pred_mask > 0.5).astype(np.uint8)
    mask_applied = pred_mask_bin * img

    return mask_applied

In [43]:
def crop_to_bounding_box(img):
    if img.ndim == 3:
        mask = np.any(img != 0, axis=2)
    else:
        mask = img != 0

    coords = np.argwhere(mask)
    if coords.size == 0:
        raise ValueError("Na obrazku nie ma postaci (wszystko czarne)!")
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1  # +1 bo końcowy indeks jest nieinclusive

    return img[y0:y1, x0:x1]

In [85]:
def show_imgs(img1, img2):
    import matplotlib.pyplot as plt

    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(img1)
    axs[0].set_title('Obraz 1')
    axs[0].axis('off')

    axs[1].imshow(img2)
    axs[1].set_title('Obraz 2')
    axs[1].axis('off')

    plt.tight_layout()
    plt.show()

In [95]:
def are_characters_identical_bbox(bbox1, bbox2, threshold=0.80):
    bbox1 = np.array(bbox1)
    bbox2 = np.array(bbox2)

    show_imgs(bbox1, bbox2)

    if bbox1.ndim == 3:
        bbox1_gray = cv2.cvtColor(bbox1, cv2.COLOR_BGR2GRAY)
    else:
        bbox1_gray = bbox1
    if bbox2.ndim == 3:
        bbox2_gray = cv2.cvtColor(bbox2, cv2.COLOR_BGR2GRAY)
    else:
        bbox2_gray = bbox2

    h1, w1 = bbox1_gray.shape
    h2, w2 = bbox2_gray.shape
    if h1 <= h2 and w1 <= w2:
        template, search_img = bbox1_gray, bbox2_gray
    elif h2 <= h1 and w2 <= w1:
        template, search_img = bbox2_gray, bbox1_gray
    else:
        new_h = max(h1, h2)
        new_w = max(w1, w2)
        def pad_to_shape(img, target_h, target_w):
            pad_h = target_h - img.shape[0]
            pad_w = target_w - img.shape[1]
            pad_top = pad_h // 2
            pad_bottom = pad_h - pad_top
            pad_left = pad_w // 2
            pad_right = pad_w - pad_left
            return np.pad(img, ((pad_top, pad_bottom), (pad_left, pad_right)), mode='constant', constant_values=0)
        bbox1_padded = pad_to_shape(bbox1_gray, new_h, new_w)
        bbox2_padded = pad_to_shape(bbox2_gray, new_h, new_w)
        template, search_img = bbox1_padded, bbox2_padded
    result = cv2.matchTemplate(search_img, template, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, _ = cv2.minMaxLoc(result)
    return max_val >= threshold

In [96]:
def compare_two(img1, img2):
    extracted1 = extract_foreground(img1)
    extracted2 = extract_foreground(img2)
    bbox1 = crop_to_bounding_box(extracted1)
    bbox2 = crop_to_bounding_box(extracted2)

    return are_characters_identical_bbox(bbox1, bbox2)

In [97]:
def segmentate_captcha(captcha_img):
    h, w = captcha_img.shape[:2]
    cell_w = w // 6
    cell_h = h // 2

    segments = []
    for row in range(2):
        for col in range(6):
            x0 = col * cell_w
            y0 = row * cell_h
            segment = captcha_img[y0:y0+cell_h, x0:x0+cell_w]
            segments.append(segment)

    pairs = {
        'A': (segments[0], segments[3]),   # A: 1 i 7
        'B': (segments[1], segments[4]),   # B: 2 i 8
        'C': (segments[2], segments[5]),   # C: 3 i 9
        'D': (segments[6], segments[9]),   # D: 4 i 10
        'E': (segments[7], segments[10]),  # E: 5 i 11
        'F': (segments[8], segments[11]),  # F: 6 i 12
    }
    return pairs

In [98]:
def solve_captcha(captcha_img):
    pairs = segmentate_captcha(captcha_img)
    results = []

    for k, v in pairs.items():
        img1, img2 = v
        identical = compare_two(img1, img2)
        if identical:
            results.append(k)

    return results

In [ ]:
if __name__ == "__main__":
    captcha_img = cv2.imread('../data/captcha/game_probes/s10.jpg')
    if captcha_img is None:
        raise ValueError("Nie można wczytać obrazu captcha_example.png")

    result = solve_captcha(captcha_img)
    print("Wynik rozwiązania captcha:", result)
